In [12]:
import pandas as pd
import matplotlib.pyplot as plt

In [13]:
deliveries = pd.read_csv(
    'C:/Users/Admin/OneDrive/Desktop/kartik/deliveries.csv',
    skipinitialspace=True,
    on_bad_lines='skip'
)

deliveries = deliveries.loc[:, ~deliveries.columns.str.contains('^Unnamed')].copy()
deliveries.columns = ['record_id', 'month', 'route_id', 'hub', 'promised_days', 'actual_days']

routes = pd.read_csv('C:/Users/Admin/OneDrive/Desktop/kartik/routes.csv')

In [14]:
deliveries['promised_days'] = pd.to_numeric(deliveries['promised_days'], errors='coerce')
deliveries['actual_days'] = pd.to_numeric(deliveries['actual_days'], errors='coerce')
deliveries = deliveries.dropna(subset=['promised_days', 'actual_days']).copy()

In [15]:
deliveries = deliveries.drop_duplicates()

In [16]:
df = deliveries.merge(routes, on='route_id', how='left')

In [17]:
df['delay_days'] = (df['actual_days'] - df['promised_days']).clip(lower=0)

In [18]:
summary = (
    df.groupby('service_type', as_index=False)
    .agg(total_delay_days=('delay_days', 'sum'))
)

incidence = (
    df.groupby('service_type')
      .apply(lambda g: (g['actual_days'] > g['promised_days']).mean() * 100)
      .reset_index(name='delay_incidence_rate')
)

summary = summary.merge(incidence, on='service_type', how='left')

route_totals = df.groupby('route_id')['delay_days'].sum()
top_route = route_totals.idxmax()
top_route_delay = route_totals.max()
total_delay = float(df['delay_days'].sum())
top_route_share = (top_route_delay / total_delay * 100) if total_delay else 0.0

print(summary.to_string(index=False))
print(f"\nTop route: {top_route}")
print(f"Top route delay: {top_route_delay}")
print(f"Total delay: {total_delay}")
print(f"Top route share: {top_route_share:.2f}%")

service_type  total_delay_days  delay_incidence_rate
     Express              12.0             66.666667
    Standard              22.0             83.333333

Top route: R4
Top route delay: 14.0
Total delay: 34.0
Top route share: 41.18%


C:\Users\Admin\AppData\Local\Temp\ipykernel_5436\3715051551.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g['actual_days'] > g['promised_days']).mean() * 100)
